# Aula 18: Proteger Agentes de IA com Recibos Criptográficos

## Caderno Prático

Este caderno orienta através de quatro exercícios:

1. **Assine o seu primeiro recibo** para uma chamada de ferramenta de agente e verifique-o.
2. **Manipule o recibo** e veja a verificação falhar.
3. **Construa uma corrente de três recibos** e confirme a integridade da cadeia.
4. **Envolva uma chamada de ferramenta do Microsoft Agent Framework** para que cada ação emita um recibo.

Todas as primitivas criptográficas são importadas de bibliotecas bem mantidas (`pynacl` para Ed25519, `jcs` para JSON canónico RFC 8785, `hashlib` da biblioteca padrão Python para SHA-256). A lógica do recibo em si é Python simples que pode ler e modificar.

Execute as células pela ordem. Cada seção é curta e autónoma.


## Configuração

Instale as duas dependências. Ambas têm licenças permissivas (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilitários Auxiliares

Estes dois auxiliares tratam da codificação base64url (sem preenchimento) e do hashing SHA-256 de objetos arbitrários. Eles mantêm o resto do notebook focado na própria lógica do recibo.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Secção 1: Assine o seu primeiro recibo

Imagine que o nosso agente da **Contoso Travel** acabou de pesquisar voos de Sydney para Los Angeles para um cliente. Queremos registar esta chamada à ferramenta como um recibo assinado para que um auditor futuro o possa verificar sem ter de confiar em nós.

### Passo 1.1: Gerar uma chave de assinatura

Em produção, a chave de assinatura do agente seria armazenada num módulo de segurança hardware (HSM), Azure Key Vault ou num repositório protegido semelhante. Para esta lição, geramos uma chave nova na memória.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Construir a payload do recibo

A payload contém tudo aquilo a que queremos que o recibo ateste: quem agiu, qual a ferramenta, com que argumentos, o que foi devolvido, sob que política, e quando. Encriptamos os argumentos e o resultado em hash em vez de os incluir inline para que o recibo não revele conteúdo sensível.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: Assinar e montar o recibo

Três etapas:

1. Canonicalizar o payload usando JCS para que duas implementações que produzam o mesmo recibo lógico produzam bytes idênticos.
2. Fazer hash dos bytes canónicos com SHA-256.
3. Assinar o hash com a chave privada Ed25519.

A assinatura é então anexada ao payload original para produzir o recibo final.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Passo 1.4: Verificar o recibo

A verificação inverte o processo. Removemos a assinatura, recalculamos o hash canónico, e verificamos a assinatura em relação à chave pública no recibo.

Um auditor que faça esta verificação não precisa de nada de nós além do próprio recibo. Nenhum serviço para chamar, nenhum diretório de chaves para consultar, nenhuma confiança necessária.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Deve ver `Receipt is valid: True`. O agente produziu o seu primeiro registo de auditoria com assinatura criptográfica.


## Secção 2: Manipular o recibo

O objetivo dos recibos é que sejam evidentes se forem manipulados. Vamos provar isso.

Vamos modificar exatamente um carácter do recibo e observar a falha na verificação.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### O que acaba de acontecer?

Quando alterámos `policy_id`, os bytes canónicos mudaram. O hash SHA-256 desses bytes mudou. A assinatura (que era sobre o hash original) já não corresponde ao novo hash. A verificação retorna corretamente `False`.

Não há forma de modificar qualquer campo do recibo e ainda assim fazer com que este verifique, a menos que o invasor tenha a chave privada. Enquanto a chave privada estiver num key vault e a chave pública estiver publicada, é impossível esconder a adulteração.

Experimente você mesmo: modifique o `tool_name` ou `agent_id` ou `timestamp` na célula acima e execute novamente. Cada alteração produz um recibo inválido.


## Section 3: Encadear recibos

Um único recibo protege uma ação. A maioria dos agentes toma muitas ações. Para tornar toda a sequência evidente de adulteração, ligamos cada recibo ao anterior incluindo o hash do recibo anterior no conteúdo do novo recibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Se alguém remover ou reordenar um recibo, a cadeia quebra exatamente nesse ponto. A verificação de qualquer recibo posterior falha porque o seu `previous_receipt_hash` deixa de corresponder ao hash real do seu predecessor.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Agora parta a cadeia ao adulterar o recibo do meio e verifique novamente. O recibo adulterado falha na verificação da assinatura, E o recibo seguinte falha na verificação da ligação da cadeia (porque o seu `previous_receipt_hash` já não corresponde ao hash do recibo do meio modificado).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

O recibo 0 continua a verificar (não foi modificado e não tem um antecessor de que dependa). O recibo 1 falha na verificação da assinatura porque alterámos `tool_args_hash`. O recibo 2 falha na verificação da cadeia de conexões porque o seu `previous_receipt_hash` foi calculado com base no recibo 1 original (agora modificado).

Mesmo que um atacante volte a assinar o recibo 1 modificado (o que não pode fazer sem a chave privada), a incompatibilidade na cadeia de conexões do recibo 2 ainda revelaria a adulteração. Para ocultar a alteração, o atacante teria de voltar a assinar todos os recibos a partir do ponto de modificação, o que requer a posse da chave privada.


## Secção 4: Envolver uma chamada de ferramenta de agente com assinatura de recibo

Numa implementação real, não quer que cada autor de agente se lembre de chamar `make_receipt`. Quer que a assinatura do recibo seja automática para cada invocação de ferramenta.

Aqui está o padrão mais simples: uma classe wrapper que recebe qualquer função de ferramenta chamada e retorna uma versão que emite recibo. Isto adapta-se a qualquer framework de agente, incluindo o Microsoft Agent Framework (`agent_framework.azure`).

Se não tiver um projeto Azure AI Foundry configurado, o mock local abaixo ainda demonstra o padrão.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integração com o Microsoft Agent Framework

O wrapper `ReceiptedTool` acima é independente do framework. Para o usar dentro de um agente construído com o Microsoft Agent Framework, registe a função envolvida como uma ferramenta. Um esboço (você substituiria o mock por um registo real de ferramenta do Azure AI Foundry):

```python
# Pseudocódigo a mostrar a forma de integração.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Você é um agente da Contoso Travel ...",
#     tools=[receipted_lookup],   # a ferramenta envolvida, não a função crua
# )
# response = agent.run("Encontra voos de Sydney para Los Angeles em junho.")
#
# # Após a execução, cada chamada de ferramenta que o agente fez tem um recibo assinado:
# audit_chain = receipted_lookup.receipts
```

O framework do agente não precisa de saber nada sobre recibos. A assinatura do recibo está envolvida na ferramenta, não integrada no framework. É assim que se adiciona proveniência ao código existente do agente sem reescrever o agente.


## Recapitulação e desafio alargado

Você:

- Gerou um par de chaves Ed25519.
- Construiu e assinou um recibo para uma chamada de ferramenta do agente.
- Verificou o recibo offline usando apenas a chave pública.
- Manipulou um recibo e observou falha na verificação.
- Construíu uma sequência encadeada por hash de três recibos.
- Manipulou o meio da cadeia e observou falha tanto na assinatura como na ligação da cadeia.
- Envolveu uma função de ferramenta com assinatura automática de recibos.

**Desafio alargado.** Estenda o esquema do recibo com um campo `request_id` (um UUID para rastreamento distribuído). Atualize `make_receipt` para o incluir, e confirme que os recibos ainda verificam de ponta a ponta. Depois modifique o campo após a assinatura e confirme que a verificação falha. Isto obriga-o a interiorizar como cada byte da codificação canónica contribui para a assinatura.

**Limite importante.** Os recibos provam três coisas e apenas três coisas: atribuição (esta chave assinou este conteúdo), integridade (o conteúdo não mudou desde a assinatura), e ordenação (este recibo veio depois daquele recibo). Eles NÃO provam que a ação do agente foi correta, que a política nomeada em `policy_id` foi realmente avaliada, ou que o agente seguiu todas as regras. Os recibos são a base. A governação é o sistema que constrói por cima.

Leia novamente o README da lição com esse limite em mente. O erro mais comum que as equipas cometem com recibos é assumir que "temos recibos" significa "estamos governados." Não significa. Os recibos tornam o comportamento do agente auditável. Não o tornam correcto.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
